<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 자연어 처리와 LLM · 08. LoRA

## LoRA: Low-Rank Adaptation of Large Language Models

- **원 논문:** [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685)
- **저자 / 발표:** Edward J. Hu et al. · ICLR 2022 (2021)
- **난이도 / 예상 시간:** 중급 · 약 120분
- **선수 지식:** Python, PyTorch, 확률·선형대수·sequence model의 기초
- **로컬 학습 데이터:** `data/field_curriculum/nlp_corpus.json`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** frozen W0 + BA low-rank update와 trainable parameter 절감을 재현한다.

**축소하거나 생략한 부분:** GPT-3 attention projection 대신 bag-of-words classifier의 rank-2 projection을 쓴다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §4.1 Eq. (3): h=W0x+BAx | Task 1 LoRALinear | forward equality |
| §4.1: W0 frozen, A/B trainable | Task 1–2 | requires_grad assert |
| §4.2: applying LoRA to Transformer | Task 2 classifier projection | rank=2 |
| §7.1: low intrinsic rank | Task 3 | parameter ratio/loss plot |

## 핵심 아이디어와 수식

$$h=W_0x+\Delta Wx=W_0x+\frac{\alpha}{r}BAx$$

**기호 해설:** W0는 frozen weight, A/B는 rank r factor, α/r은 update scale이다.

**코드 대응:** Task 1이 LoRALinear, Task 2가 classifier, Task 3이 trainable-only 최적화를 구현한다.

**입력과 출력 shape:** bag-of-words [B,V] → LoRA hidden [B,16] → class logits [B,2].

**포트폴리오 구현 범위:** Transformer Q/V 대신 작은 projection에 적용하지만 low-rank/freeze 계약은 동일하다.

각 단계는 데이터 전처리와 shape 확인부터 논문 고유 class, `forward`, 목적함수,
`train_step`, 평가까지 이어진다. 핵심 수식을 한 번의 고수준 호출로 감추지 않고
중간 tensor를 확인할 수 있게 분리한다.

In [ ]:
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
import json
import math
import re
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def locate(relative_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


dataset_path = locate("data/field_curriculum/nlp_corpus.json")
corpus = json.loads(dataset_path.read_text(encoding="utf-8"))
sentences = corpus["sentences"]
translation_pairs = corpus["translation_pairs"]
documents = corpus["documents"]
queries = corpus["queries"]
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<mask>", "<sentinel>"]
sentence_words = {word.lower() for sentence in sentences for word in sentence.split()}
translation_words = {
    word.lower()
    for pair in translation_pairs
    for side in ("source", "target")
    for word in pair[side].split()
}
words = sorted(sentence_words | translation_words)
itos = SPECIAL_TOKENS + words
stoi = {word: index for index, word in enumerate(itos)}
PAD, BOS, EOS, MASK, SENTINEL = range(5)
VOCAB_SIZE = len(itos)


def encode(text, length=4):
    token_ids = [stoi[word.lower()] for word in text.split()]
    token_ids = token_ids[:length]
    return token_ids + [PAD] * (length - len(token_ids))


sentence_ids = torch.tensor([encode(text) for text in sentences])
sentiment_labels = torch.tensor(corpus["sentiment_labels"], dtype=torch.long)
source_ids = torch.tensor([encode(pair["source"], 3) for pair in translation_pairs])
target_ids = torch.tensor([encode(pair["target"], 3) for pair in translation_pairs])
sentence_ids, sentiment_labels = ACCELERATOR.move(
    sentence_ids,
    sentiment_labels,
)
source_ids, target_ids = ACCELERATOR.move(source_ids, target_ids)
assert sentence_ids.shape == (360, 4)
assert source_ids.shape == target_ids.shape == (60, 3)
print(ACCELERATOR.summary())
print(dataset_path.name, f"sentences={len(sentences)}, vocab={VOCAB_SIZE}")

### 구현 단계 1 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §4.1 Eq. (3): h=W0x+BAx
- **노트북에서 구현할 부분:** Task 1 LoRALinear
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** forward equality

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
@dataclass
class LoRAConfig:
    rank: int = 2
    alpha: float = 2.0
    hidden_dim: int = 16
    learning_rate: float = 0.08
    steps: int = 28


class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, config):
        super().__init__()
        base = torch.randn(out_features, in_features) * 0.02
        self.base_weight = nn.Parameter(base, requires_grad=False)
        self.factor_a = nn.Parameter(torch.randn(config.rank, in_features) * 0.02)
        self.factor_b = nn.Parameter(torch.zeros(out_features, config.rank))
        self.scale = config.alpha / config.rank

    def low_rank_update(self, inputs):
        projected = F.linear(inputs, self.factor_a)
        return self.scale * F.linear(projected, self.factor_b)

    def forward(self, inputs):
        base = F.linear(inputs, self.base_weight)
        return base + self.low_rank_update(inputs)


config = LoRAConfig()
layer = LoRALinear(5, 2, config).to(DEVICE)
probe = torch.randn(3, 5, device=DEVICE)
assert torch.allclose(layer(probe), F.linear(probe, layer.base_weight))
assert not layer.base_weight.requires_grad

### 구현 단계 2 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §4.1: W0 frozen, A/B trainable
- **노트북에서 구현할 부분:** Task 1–2
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** requires_grad assert

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
def build_bag_of_words(texts, vocabulary, word_to_id, device):
    features = torch.zeros(len(texts), len(vocabulary), device=device)
    for row, text in enumerate(texts):
        for word in text.lower().split():
            features[row, word_to_id[word]] += 1
    return features


class LoRAClassifier(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        self.adaptation = LoRALinear(vocab_size, config.hidden_dim, config)
        self.classifier = nn.Linear(config.hidden_dim, 2)

    def forward(self, features):
        hidden = torch.tanh(self.adaptation(features))
        return self.classifier(hidden)


features = build_bag_of_words(sentences, itos, stoi, DEVICE)
model = LoRAClassifier(VOCAB_SIZE, config).to(DEVICE)
assert features.shape == (360, VOCAB_SIZE)
assert model(features[:2]).shape == (2, 2)

### 구현 단계 3 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §7.1: low intrinsic rank
- **노트북에서 구현할 부분:** Task 3
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** parameter ratio/loss plot

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
def lora_classification_loss(logits, targets):
    return F.cross_entropy(logits, targets)


def train_lora_step(model, optimizer, features, targets):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss = lora_classification_loss(model(features), targets)
    loss.backward()
    optimizer.step()
    return float(loss.detach())


def evaluate_lora(model, features, targets):
    model.eval()
    with torch.no_grad():
        predictions = model(features).argmax(dim=1)
    return float((predictions == targets).float().mean())


trainable = [parameter for parameter in model.parameters() if parameter.requires_grad]
optimizer = torch.optim.Adam(trainable, lr=config.learning_rate)
losses = []
for step in range(config.steps):
    losses.append(
        train_lora_step(
            model,
            optimizer,
            features[:240],
            sentiment_labels[:240],
        )
    )
accuracy = evaluate_lora(
    model,
    features[240:],
    sentiment_labels[240:],
)
trainable_count = sum(parameter.numel() for parameter in trainable)
full_count = model.adaptation.base_weight.numel()
assert model.adaptation.base_weight.grad is None
assert trainable_count < full_count
assert min(losses[3:]) < losses[0]
figure, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].plot(losses)
axes[0].set_title("LoRA adaptation loss")
axes[1].bar(["trainable", "full W"], [trainable_count, full_count])
axes[1].set_title(f"parameter count, acc={accuracy:.1%}")
plt.tight_layout()
plt.show()

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> GPT-3 attention projection 대신 bag-of-words classifier의 rank-2 projection을 쓴다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.